# Tree Crown Tracking Across Orthomosaics

This notebook implements a graph-based approach to track tree crowns across multiple orthomosaics (OMs) using spatial alignment and similarity matching.

## Key Features:
- **TreeTrackingGraph class**: Complete tracking pipeline with configurable matching parameters
- **Spatial alignment**: Consecutive alignment to minimize positional drift
- **Multi-case matching**: one_to_one, containment, and nearby cases
- **Image extraction**: RGB patches extracted from original (unshifted) geometries
- **Comprehensive visualization**: Chain tracking with both polygons and extracted images

## Workflow:
1. Load crown geometries and orthomosaics
2. Extract RGB images from **original** crown positions
3. Apply spatial alignment (shift geometries)
4. Build matching graph with relaxed parameters
5. Analyze and visualize tracking chains

## 1. Imports and Setup

In [ ]:
import os
import re
import json
from dataclasses import dataclass, replace
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple, Iterable

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping, Polygon
from shapely.affinity import translate
import networkx as nx

## 2. TreeTrackingGraph Class

The main class for tree crown tracking across orthomosaics.

In [ ]:
@dataclass
class MatchCaseConfig:
    """Configuration for a specific matching case type."""
    name: str
    base_similarity_weights: Dict[str, float]
    scoring_weights: Dict[str, float]
    similarity_threshold: float
    min_iou: float = 0.0
    min_overlap_prev: float = 0.0
    min_overlap_curr: float = 0.0
    max_centroid_dist: Optional[float] = None
    mutual_best: bool = False
    allow_multiple: bool = False
    max_edges_per_prev: Optional[int] = None
    max_edges_per_curr: Optional[int] = None


class TreeTrackingGraph:
    """
    Tree crown tracker across orthomosaics using directed graph.
    
    Features:
    - Discovers crown GeoPackages and orthomosaics
    - Extracts RGB images from ORIGINAL (unshifted) crowns
    - Applies spatial alignment for matching
    - Builds match graph with configurable case-based matching
    - Reports quality metrics and graph complexity
    """
    
    def __init__(
        self,
        crown_dir: Optional[str] = None,
        ortho_dir: Optional[str] = None,
        output_dir: str = '../../output',
        simplify_tol: float = 1.0,
        max_crowns_preview: int = 200,
        auto_discover: bool = True
    ) -> None:
        self.output_dir = output_dir
        self.simplify_tol = simplify_tol
        self.max_crowns_preview = max_crowns_preview
        self.crown_dir = crown_dir or self._resolve_dir('input/input_crowns', '../../input/input_crowns')
        self.ortho_dir = ortho_dir or self._resolve_dir('input/input_om', '../../input/input_om')
        
        self.file_pairs: List[Tuple[str, Optional[str]]] = []
        self.om_ids: List[int] = []
        self.crowns_gdfs: Dict[int, gpd.GeoDataFrame] = {}
        self.crown_attrs: Dict[int, List[Dict[str, Any]]] = {}
        self.crown_images: Dict[int, List[Optional[np.ndarray]]] = {}  # Store ORIGINAL images
        self.G: nx.DiGraph = nx.DiGraph()
        
        self.case_configs: Dict[str, MatchCaseConfig] = self._ultra_relaxed_case_configs()
        self.case_order: List[str] = ['one_to_one', 'containment', 'nearby']
        self.last_case_counts: Dict[str, int] = {}
        self.last_selected_counts: Dict[str, int] = {}
        
        if auto_discover:
            self.discover_files()
    
    @staticmethod
    def _resolve_dir(root_rel: str, nb_rel: str) -> str:
        """Resolve directory path from multiple candidates."""
        candidates = [
            os.path.abspath(os.path.join(os.getcwd(), root_rel)),
            os.path.abspath(os.path.join(os.getcwd(), nb_rel)),
        ]
        for p in candidates:
            if os.path.isdir(p):
                return p
        raise FileNotFoundError(f"Could not resolve directory for {root_rel}. Tried: {candidates}")
    
    @staticmethod
    def _extract_numeric_id(name: str) -> Optional[int]:
        """Extract numeric ID from filename."""
        m = re.search(r"(\d+)", os.path.basename(name))
        return int(m.group(1)) if m else None
    
    def discover_files(self) -> None:
        """Discover crown and orthomosaic files from directories."""
        crown_files = [
            os.path.join(self.crown_dir, f) 
            for f in os.listdir(self.crown_dir) 
            if f.lower().endswith('.gpkg')
        ]
        ortho_files = []
        if os.path.exists(self.ortho_dir):
            ortho_files = [
                os.path.join(self.ortho_dir, f) 
                for f in os.listdir(self.ortho_dir) 
                if f.lower().endswith('.tif')
            ]
        
        if not crown_files:
            raise FileNotFoundError(f"No .gpkg crown files found in {self.crown_dir}")
        
        # Match crown files with orthomosaics by numeric ID
        crowns_by_id = {self._extract_numeric_id(cf): cf for cf in crown_files}
        orthos_by_id = {self._extract_numeric_id(of): of for of in ortho_files}
        
        numeric_ids = sorted(
            set(k for k in crowns_by_id.keys() if isinstance(k, int)) & 
            set(k for k in orthos_by_id.keys() if isinstance(k, int))
        )
        
        file_pairs: List[Tuple[str, Optional[str]]] = []
        om_ids: List[int] = []
        
        for nid in numeric_ids:
            file_pairs.append((crowns_by_id[nid], orthos_by_id.get(nid)))
            om_ids.append(nid)
        
        # Sort by OM ID
        pairs_with_id = sorted(zip(om_ids, file_pairs), key=lambda x: x[0])
        self.om_ids = [oid for oid, _ in pairs_with_id]
        self.file_pairs = [fp for _, fp in pairs_with_id]

In [ ]:
    def load_data(self, load_images: bool = False) -> None:
        """
        Load crown data from GeoPackages and optionally extract image patches.
        
        IMPORTANT: Images are extracted from ORIGINAL (unshifted) geometries.
        The geometries are then shifted for spatial alignment, but images remain
        from the original positions to match the orthomosaic data.
        
        Args:
            load_images: If True, extract RGB image patches from orthomosaics
        """
        self.crowns_gdfs.clear()
        self.crown_attrs.clear()
        self.crown_images.clear()
        
        for om_id, (crown_file, ortho_file) in zip(self.om_ids, self.file_pairs):
            # Load crown geometries
            gdf = gpd.read_file(crown_file)
            
            # CRITICAL FIX: Extract images BEFORE any shifting
            if load_images and ortho_file and os.path.exists(ortho_file):
                print(f"Extracting images for OM{om_id} from {os.path.basename(ortho_file)}...")
                with rasterio.open(ortho_file) as src:
                    patches: List[Optional[np.ndarray]] = []
                    for idx, row in gdf.iterrows():
                        geom = [mapping(row.geometry)]  # Use ORIGINAL geometry
                        try:
                            out_image, _ = mask(src, geom, crop=True, all_touched=True)
                            # Convert from (bands, height, width) to (height, width, bands)
                            img_patch = np.moveaxis(out_image[:3, :, :], 0, -1)  # First 3 bands (RGB)
                            # Normalize to 0-1 if needed
                            if img_patch.max() > 1:
                                img_patch = img_patch.astype(np.float32) / img_patch.max()
                        except Exception as e:
                            print(f"  Warning: Could not extract image for crown {idx}: {e}")
                            img_patch = None
                        patches.append(img_patch)
                    self.crown_images[om_id] = patches
                    print(f"  Extracted {sum(p is not None for p in patches)}/{len(patches)} images")
            else:
                self.crown_images[om_id] = [None] * len(gdf)
            
            # Store original GeoDataFrames (will be replaced with aligned versions later)
            self.crowns_gdfs[om_id] = gdf
            
            # Compute attributes for ORIGINAL geometries
            self.crown_attrs[om_id] = [
                self._compute_crown_attributes(row.geometry) 
                for _, row in gdf.iterrows()
            ]
        
        print(f"\n✓ Loaded {sum(len(gdf) for gdf in self.crowns_gdfs.values())} crowns from {len(self.om_ids)} OMs")
    
    @staticmethod
    def _compute_crown_attributes(geometry) -> Dict[str, Any]:
        """Compute geometric attributes for a crown polygon."""
        centroid = geometry.centroid
        area = geometry.area
        perimeter = geometry.length
        bounds = geometry.bounds
        
        # Compactness (circularity)
        compactness = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
        
        # Eccentricity (using minimum rotated rectangle)
        try:
            min_rect = geometry.minimum_rotated_rectangle
            coords = list(min_rect.exterior.coords)
            side1 = np.linalg.norm(np.array(coords[0]) - np.array(coords[1]))
            side2 = np.linalg.norm(np.array(coords[1]) - np.array(coords[2]))
            major_axis = max(side1, side2)
            minor_axis = min(side1, side2)
            eccentricity = minor_axis / major_axis if major_axis > 0 else 1
        except Exception:
            eccentricity = 1
        
        # Aspect ratio (bounding box)
        aspect_ratio = (
            (bounds[3] - bounds[1]) / (bounds[2] - bounds[0]) 
            if bounds[2] != bounds[0] else 1
        )
        
        return {
            'geometry': geometry,
            'centroid': centroid,
            'area': area,
            'perimeter': perimeter,
            'compactness': compactness,
            'eccentricity': eccentricity,
            'aspect_ratio': aspect_ratio,
            'bounds': bounds,
        }

In [ ]:
    def _ultra_relaxed_case_configs(self) -> Dict[str, MatchCaseConfig]:
        """
        Ultra-relaxed matching parameters optimized for challenging OM transitions.
        
        Key features:
        - Low similarity thresholds (0.25 for one_to_one, 0.20 for nearby)
        - Heavily weighted towards spatial proximity (70-85%) over shape
        - Minimal overlap requirements (5-10%)
        - Larger distance thresholds (150-200m)
        - Non-mutual matching to capture more edges
        """
        return {
            'one_to_one': MatchCaseConfig(
                name='one_to_one',
                base_similarity_weights={
                    'spatial': 0.70,  # Prioritize spatial proximity
                    'area': 0.15,
                    'shape': 0.05,    # De-emphasize shape (phenology changes)
                    'iou': 0.10
                },
                scoring_weights={
                    'base': 0.60,
                    'iou': 0.10,
                    'overlap_prev': 0.05,
                    'overlap_curr': 0.05,
                    'centroid': 0.20
                },
                similarity_threshold=0.25,  # Very relaxed
                min_iou=0.01,               # Just 1% overlap
                min_overlap_prev=0.10,      # Just 10% overlap
                min_overlap_curr=0.10,
                max_centroid_dist=150.0,    # Larger distance
                mutual_best=False,          # Allow non-mutual matches
                allow_multiple=False,
                max_edges_per_prev=1,
                max_edges_per_curr=1,
            ),
            'containment': MatchCaseConfig(
                name='containment',
                base_similarity_weights={
                    'spatial': 0.50,
                    'area': 0.20,
                    'shape': 0.10,
                    'iou': 0.20
                },
                scoring_weights={
                    'base': 0.40,
                    'overlap_prev': 0.30,
                    'overlap_curr': 0.30
                },
                similarity_threshold=0.25,
                min_iou=0.0,
                min_overlap_prev=0.25,
                min_overlap_curr=0.25,
                max_centroid_dist=150.0,
                mutual_best=False,
                allow_multiple=False,
                max_edges_per_prev=1,
                max_edges_per_curr=1,
            ),
            'nearby': MatchCaseConfig(
                name='nearby',
                base_similarity_weights={
                    'spatial': 0.85,  # Almost purely spatial
                    'area': 0.10,
                    'shape': 0.03,
                    'iou': 0.02
                },
                scoring_weights={
                    'base': 0.70,
                    'centroid': 0.30
                },
                similarity_threshold=0.20,  # Ultra relaxed
                min_iou=0.0,
                min_overlap_prev=0.05,
                min_overlap_curr=0.05,
                max_centroid_dist=200.0,    # Very large distance
                mutual_best=False,
                allow_multiple=False,
                max_edges_per_prev=1,
                max_edges_per_curr=1,
            ),
        }
    
    @staticmethod
    def _compute_iou(g1, g2) -> float:
        """Compute Intersection over Union."""
        try:
            intersection = g1.intersection(g2).area
            union = g1.union(g2).area
        except Exception:
            intersection = 0.0
            union = g1.area + g2.area
        return intersection / union if union > 0 else 0.0
    
    def _weighted_similarity(
        self,
        a1: Dict[str, Any],
        a2: Dict[str, Any],
        weights: Optional[Dict[str, float]] = None,
        max_dist: float = 100.0
    ) -> Tuple[float, Dict[str, float]]:
        """Compute weighted similarity between two crowns."""
        if weights is None:
            weights = {'spatial': 0.4, 'area': 0.2, 'shape': 0.2, 'iou': 0.2}
        
        # Spatial similarity (based on centroid distance)
        centroid_dist = a1['centroid'].distance(a2['centroid'])
        spatial_sim = max(0.0, 1.0 - (centroid_dist / max_dist))
        
        # Area similarity
        area_sim = (
            min(a1['area'], a2['area']) / max(a1['area'], a2['area']) 
            if max(a1['area'], a2['area']) > 0 else 0.0
        )
        
        # Shape similarity (compactness + eccentricity)
        compactness_sim = 1.0 - abs(a1['compactness'] - a2['compactness'])
        eccentricity_sim = 1.0 - abs(a1['eccentricity'] - a2['eccentricity'])
        shape_sim = (compactness_sim + eccentricity_sim) / 2.0
        
        # IoU similarity
        iou_sim = self._compute_iou(a1['geometry'], a2['geometry'])
        
        # Weighted total
        total = (
            weights.get('spatial', 0.0) * spatial_sim +
            weights.get('area', 0.0) * area_sim +
            weights.get('shape', 0.0) * shape_sim +
            weights.get('iou', 0.0) * iou_sim
        )
        
        return total, {
            'spatial': float(spatial_sim),
            'area': float(area_sim),
            'shape': float(shape_sim),
            'iou': float(iou_sim),
            'total': float(total)
        }

In [ ]:
    def _compute_pair_metrics(
        self,
        prev_attrs: Dict[str, Any],
        curr_attrs: Dict[str, Any],
        max_dist: float
    ) -> Dict[str, float]:
        """Compute all pairwise metrics between two crowns."""
        prev_geom = prev_attrs['geometry']
        curr_geom = curr_attrs['geometry']
        
        # Intersection and union
        try:
            intersection_area = prev_geom.intersection(curr_geom).area
        except Exception:
            intersection_area = 0.0
        
        try:
            union_area = prev_geom.union(curr_geom).area
        except Exception:
            union_area = prev_attrs['area'] + curr_attrs['area'] - intersection_area
        
        # Overlap fractions
        prev_area = prev_attrs['area'] if prev_attrs['area'] > 0 else 1e-6
        curr_area = curr_attrs['area'] if curr_attrs['area'] > 0 else 1e-6
        overlap_prev = intersection_area / prev_area
        overlap_curr = intersection_area / curr_area
        iou = intersection_area / union_area if union_area > 0 else 0.0
        
        # Distance
        centroid_dist = prev_attrs['centroid'].distance(curr_attrs['centroid'])
        
        # Base similarity
        base_similarity, parts = self._weighted_similarity(prev_attrs, curr_attrs, max_dist=max_dist)
        
        # Additional metrics
        prev_radius = np.sqrt(prev_area / np.pi)
        curr_radius = np.sqrt(curr_area / np.pi)
        mean_radius = max((prev_radius + curr_radius) / 2.0, 1e-3)
        
        area_ratio = curr_area / prev_area if prev_area > 0 else np.inf
        if not np.isfinite(area_ratio) or area_ratio <= 0:
            balanced_area_ratio = 0.0
        else:
            balanced_area_ratio = area_ratio if area_ratio <= 1 else 1 / area_ratio
        
        # Containment tests
        try:
            prev_contains_curr = prev_geom.buffer(0).contains(curr_geom)
        except Exception:
            prev_contains_curr = False
        
        try:
            curr_contains_prev = curr_geom.buffer(0).contains(prev_geom)
        except Exception:
            curr_contains_prev = False
        
        return {
            'intersection_area': float(intersection_area),
            'union_area': float(union_area),
            'overlap_prev': float(overlap_prev),
            'overlap_curr': float(overlap_curr),
            'iou': float(iou),
            'centroid_dist': float(centroid_dist),
            'base_similarity': float(base_similarity),
            'spatial_similarity': float(parts['spatial']),
            'area_similarity': float(parts['area']),
            'shape_similarity': float(parts['shape']),
            'mean_radius': float(mean_radius),
            'area_ratio': float(area_ratio if np.isfinite(area_ratio) else 0.0),
            'balanced_area_ratio': float(balanced_area_ratio),
            'prev_contains_curr': bool(prev_contains_curr),
            'curr_contains_prev': bool(curr_contains_prev),
        }
    
    def _classify_match_case(
        self,
        prev_node: Tuple[int, int],
        curr_node: Tuple[int, int],
        features: Dict[str, float],
        prev_overlap_counts: Dict[Tuple[int, int], int],
        curr_overlap_counts: Dict[Tuple[int, int], int],
        overlap_gate: float
    ) -> str:
        """
        Classify a candidate match into a case type.
        
        Uses DYNAMIC thresholds based on overlap_gate parameter (not hardcoded).
        """
        # Containment: One crown contains the other
        if features['prev_contains_curr'] or features['curr_contains_prev']:
            return 'containment'
        
        # Extract features
        overlap_prev = features['overlap_prev']
        overlap_curr = features['overlap_curr']
        iou = features['iou']
        prev_count = prev_overlap_counts.get(prev_node, 0)
        curr_count = curr_overlap_counts.get(curr_node, 0)
        centroid_dist = features['centroid_dist']
        
        # one_to_one: Use overlap_gate parameter dynamically (not hardcoded)
        min_overlap_for_one_to_one = max(overlap_gate, 0.30)
        min_iou_for_one_to_one = max(overlap_gate / 2, 0.10)
        
        if (prev_count == 1 and curr_count == 1 and
            overlap_prev >= min_overlap_for_one_to_one and
            overlap_curr >= min_overlap_for_one_to_one and
            iou >= min_iou_for_one_to_one):
            return 'one_to_one'
        
        # nearby: Primarily spatial proximity
        if centroid_dist < 30.0 and (overlap_prev >= overlap_gate or overlap_curr >= overlap_gate):
            return 'nearby'
        
        return 'none'
    
    def _score_candidate(
        self,
        base_similarity: float,
        similarity_parts: Dict[str, float],
        features: Dict[str, float],
        config: MatchCaseConfig
    ) -> float:
        """Score a candidate match using weighted components."""
        centroid_factor = 1.0 - min(1.0, features['centroid_dist'] / (features['mean_radius'] * 3.0))
        
        components = {
            'base': base_similarity,
            'spatial': similarity_parts.get('spatial', 0.0),
            'area': similarity_parts.get('area', 0.0),
            'shape': similarity_parts.get('shape', 0.0),
            'iou': features['iou'],
            'overlap_prev': features['overlap_prev'],
            'overlap_curr': features['overlap_curr'],
            'centroid': max(0.0, centroid_factor),
            'area_balance': features.get('balanced_area_ratio', 0.0),
        }
        
        score = 0.0
        for key, weight in config.scoring_weights.items():
            score += weight * components.get(key, 0.0)
        
        return score